# 01 — Impossible Travel Investigation Workflow

Multi-agent investigation for **Impossible Travel** detections using the **Microsoft Agent Framework**.

```
                                  ┌──> [LocationAnomaly]
                                  ├──> [AuthenticationAnomaly]
                                  ├──> [TokenAnomaly]
                                  ├──> [BruteForcePattern]
                                  ├──> [MfaAbuse]
Sentinel Detection ──> Context ──┤──> [PostLoginActivity]
                       Enricher  ├──> [OAuthAbuse]
                                  ├──> [MailboxManipulation]
                                  ├──> [HighPrivilegeUser]
                                  └──> [DisabledAccountSignIn]
                                              ↓ (all 10)
                                       fan-in → Aggregator → PACO
```

| Concept | Usage |
|---|---|
| `WorkflowBuilder` | Graph-based orchestration with fan-out/fan-in |
| `Executor` | Custom deterministic steps (no LLM) for context enrichment and aggregation |
| `Agent` + `AgentExecutor` | 10 specialized risk evaluation sub-agents + PACO decision maker |
| `@tool` | Function tools for simulated data source queries |
| Structured output | `response_format=PydanticModel` for type-safe agent responses |
| DevUI | Interactive graph visualization and agent testing |

> **Prerequisite**: Run `00-setup.ipynb` first to configure Foundry and save `workshop_config.json`.

## Load Config & Imports

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import json
import time
from pathlib import Path

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

# All workflow internals (models, @tool functions, executors, ThreatDecisionPolicy,
# build_workflow) live in the shared module so 01-investigation and the deployed
# hosted_agent run identical code.
import investigation_workflow as iw
from investigation_workflow import (
    NormalizedDetection,
    ContextEnrichment,
    RiskEvidence,
    ThreatDecisionRecord,
    ActionItem,
    InvestigationVerdict,
    RiskFactorName,
    THREAT_DECISION_POLICY,
    load_test_case,
    build_workflow,
)

# Load config saved by 00-setup.ipynb
config_path = Path("workshop_config.json")
if not config_path.exists():
    raise FileNotFoundError("workshop_config.json not found. Run 00-setup.ipynb first.")

with open(config_path) as f:
    config = json.load(f)

PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = config["MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential()
af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)

print(f"✅ Connected to {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

## Data Models

In [ ]:
# Models are defined in investigation_workflow.py — already imported above.
# Listed here for reference:
print("✅ Models available (from investigation_workflow):")
for name in [
    "RiskFactorName",
    "NormalizedDetection",
    "ContextEnrichment",
    "RiskEvidence",
    "ThreatDecisionRecord",
    "ActionItem",
    "InvestigationVerdict",
]:
    print(f"   • {name}")

## Test Case & Mock Data Sources

The `test_cases/` folder contains self-contained scenarios (Sentinel detection + every
data source the 10 risk-evaluation tools query). In production the data-query helpers
below would call **Microsoft Sentinel / Entra ID / Defender** via MCP; here they read
from in-memory dicts populated by the selected test case.

**Switch test cases** by changing the `TEST_CASE` variable below. Add your own
scenario by dropping a new JSON file in `test_cases/` (see `test_cases/README.md`
for the schema).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Test case loader — selects one scenario JSON from ./test_cases/
# ═══════════════════════════════════════════════════════════════

TEST_CASES_DIR = Path("test_cases")

# 👉 Change this to switch scenarios (filename without .json),
#    or drop your own file in test_cases/ and reference it here.
TEST_CASE = "03_mfa_fatigue_attack"

# Discover available cases
available_cases = sorted(p.stem for p in TEST_CASES_DIR.glob("*.json"))
if not available_cases:
    raise FileNotFoundError(f"No test cases found under {TEST_CASES_DIR.resolve()}")

if TEST_CASE not in available_cases:
    raise ValueError(
        f"Test case '{TEST_CASE}' not found. Available: {available_cases}"
    )

case_path = TEST_CASES_DIR / f"{TEST_CASE}.json"

# load_test_case() resets and re-populates the data-source dicts inside
# investigation_workflow that the @tool functions read from.
TEST_CASE_DATA = load_test_case(case_path)

SIMULATED_USER = TEST_CASE_DATA["detection"]["PrimaryEntityId"]
DETECTION_PAYLOAD = TEST_CASE_DATA["detection"]

print(f"✅ Loaded test case: {TEST_CASE_DATA['name']}")
print(f"   File:                {case_path}")
print(f"   Description:         {TEST_CASE_DATA['description']}")
print(f"   Expected verdict:    {TEST_CASE_DATA.get('expected_verdict', '—')}"
      f" ({TEST_CASE_DATA.get('expected_confidence', '—')} confidence)")
print(f"   Target user:         {SIMULATED_USER}")
print(f"")
print(f"   Available test cases (set TEST_CASE = …):")
for name in available_cases:
    marker = "→" if name == TEST_CASE else " "
    print(f"     {marker} {name}")

# ── Detection payload for DevUI ───────────────────────────────
# Copy the JSON block below and paste it as input to the
# ImpossibleTravelInvestigation workflow in DevUI.
print("")
print("─" * 70)
print("📋 Detection payload (copy into DevUI workflow input):")
print("─" * 70)
print(json.dumps(DETECTION_PAYLOAD, indent=2))
print("")
print("   Compact (single-line) form:")
print(f"   {json.dumps(DETECTION_PAYLOAD, separators=(',', ':'))}")

In [ ]:
# 10 @tool functions and ThreatDecisionPolicy are defined in
# investigation_workflow.py. They read from the in-memory data dicts that
# load_test_case() above hydrates.
print(f"✅ 10 @tool functions and ThreatDecisionPolicy loaded from investigation_workflow")
print(f"   Risk factors: {[rf.value for rf in RiskFactorName]}")

## Agents & Custom Executors

In [ ]:
# ContextEnricher (deterministic context enrichment — no LLM) is defined in
# investigation_workflow.py and instantiated by build_workflow() below.
print("✅ ContextEnricher executor available via investigation_workflow")

In [ ]:
# Build the full investigation workflow (10 risk sub-agents + ContextEnricher +
# RiskAggregator + PACO) from the shared module. Use a separate FoundryChatClient
# for PACO so it can run on a different deployment (e.g. a larger reasoning model).
paco_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model="gpt-5.4",
    credential=credential,
)

components = build_workflow(af_client, paco_client=paco_client, store=True)

# Expose the pieces with the names the rest of the notebook expects.
context_enricher   = components.context_enricher
risk_aggregator    = components.risk_aggregator
ALL_RISK_AGENTS    = components.risk_agents
ALL_RISK_EXECUTORS = components.risk_executors
paco_agent         = components.paco_agent
paco_executor      = components.paco_executor
investigation_workflow = components.workflow

print(f"✅ Created {len(ALL_RISK_AGENTS)} specialized risk evaluation sub-agents:")
for a in ALL_RISK_AGENTS:
    print(f"   • {a.name}")
print("✅ PACO agent created (LLM-powered decision maker, model: gpt-5.4)")
print("✅ Investigation workflow built")
print("   ContextEnricher → 10 risk sub-agents (fan-out) → RiskAggregator (fan-in) → PACO")

In [ ]:
# RiskAggregator (deterministic — no LLM) is built by build_workflow() above.
# Variable `risk_aggregator` is already in scope.
print("✅ RiskAggregator executor available (instantiated by build_workflow)")

In [ ]:
# PACO is built by build_workflow() above. `paco_agent` and `paco_executor`
# are already in scope.
print("✅ PACO agent wired by build_workflow")

## Build & Run Investigation Workflow

In [ ]:
# Workflow graph was assembled in the cell above; `investigation_workflow`
# is bound to it. Confirm and continue to the run cell.
print("✅ Investigation workflow ready:", investigation_workflow.name)

In [10]:
# Run the investigation against the detection from the loaded test case.
# To try a different scenario: change TEST_CASE above and re-run the
# "Test case loader" cell, then re-run this cell.

incident = NormalizedDetection.model_validate(DETECTION_PAYLOAD)

print("═" * 70)
print(f"🚨 IMPOSSIBLE TRAVEL INVESTIGATION — {TEST_CASE_DATA['name']}")
print("═" * 70)
print(f"  User:      {incident.PrimaryEntityId}")
print(f"  Scenario:  {incident.ThreatScenario}")
print(f"  IoC IPs:   {incident.IoC_IPs}")
print(f"  Locations: {incident.IoC_Locations}")
print(f"  Severity:  {incident.Severity}")
print(f"  Expected:  {TEST_CASE_DATA.get('expected_verdict', '—')}"
      f" ({TEST_CASE_DATA.get('expected_confidence', '—')} confidence)")
print("═" * 70)

# Reset workflow state in case a previous run failed mid-execution
investigation_workflow._is_running = False

start_time = time.time()
result = await investigation_workflow.run(incident.model_dump_json())
total_time = time.time() - start_time

# Extract the final verdict from workflow outputs
final_verdict = None
for output in result.get_outputs():
    text = output.text if hasattr(output, "text") else str(output)
    try:
        final_verdict = InvestigationVerdict.model_validate_json(text)
        break
    except Exception:
        pass

print(f"\n✅ Investigation complete in {total_time:.1f}s")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
🚨 IMPOSSIBLE TRAVEL INVESTIGATION — OAuth Consent Phishing + Impossible Travel
══════════════════════════════════════════════════════════════════════
  User:      alice.morgan@contoso.com
  Scenario:  AccountCompromise
  IoC IPs:   ['203.0.113.150', '45.77.123.9']
  Locations: ['Seattle, US', 'Frankfurt, DE']
  Severity:  High
  Expected:  AccountCompromised (High confidence)
══════════════════════════════════════════════════════════════════════
  🔍 Context Enricher: Alice Morgan | Dept=Human Resources | HighPriv=False | Disabled=False
  📊 Risk Aggregator: 10 evidence records, 9 active / 10 total risk factors

✅ Investigation complete in 325.6s
══════════════════════════════════════════════════════════════════════


## Results

In [11]:
if final_verdict:
    print("═" * 70)
    print("📋 INVESTIGATION VERDICT")
    print("═" * 70)
    print(f"  Verdict:    {final_verdict.verdict}")
    print(f"  Confidence: {final_verdict.confidence}")
    print(f"  Reasoning:  {final_verdict.reasoning[:200]}")

    print(f"\n{'─' * 70}")
    print("📝 NARRATIVE")
    print(f"{'─' * 70}")
    print(final_verdict.narrative)

    if final_verdict.actionPlan:
        print(f"\n{'─' * 70}")
        print("🛡️  ACTION PLAN")
        print(f"{'─' * 70}")
        print(f"  {'#':<3} {'Action':<25} {'Target':<30} {'Destructive':<13} {'Approval'}")
        print(f"  {'─'*3} {'─'*25} {'─'*30} {'─'*13} {'─'*10}")
        for i, action in enumerate(final_verdict.actionPlan, 1):
            destructive = "⚠️  YES" if action.destructive else "No"
            approval = "🔒 REQUIRED" if action.requiresApproval else "Auto"
            print(f"  {i:<3} {action.action:<25} {action.target:<30} {destructive:<13} {approval}")

    print(f"\n{'═' * 70}")
    print("📊 FULL VERDICT JSON")
    print(f"{'═' * 70}")
    print(final_verdict.model_dump_json(indent=2))
else:
    print("⚠️  No verdict was produced. Check the workflow execution above for errors.")

══════════════════════════════════════════════════════════════════════
📋 INVESTIGATION VERDICT
══════════════════════════════════════════════════════════════════════
  Verdict:    AccountCompromised
  Confidence: High
  Reasoning:  Multiple high-scoring, corroborating indicators strongly support active account compromise rather than a benign anomaly. Evidence includes impossible travel between Seattle and Lagos, malicious/suspic

──────────────────────────────────────────────────────────────────────
📝 NARRATIVE
──────────────────────────────────────────────────────────────────────
On 2025-01-15, john.doe@contoso.com triggered a high-severity account compromise investigation. The timeline begins with anomalous sign-in activity showing impossible travel between Seattle, US and Lagos, NG within 17 minutes, with suspicious IPs 203.0.113.42 and 198.51.100.7 involved. One of the IPs showed a brute-force pattern of multiple failed attempts followed by a successful sign-in. Authentication tele

## DevUI — Interactive Testing

Launch DevUI to visualize the investigation workflow graph and
interactively test individual risk agents via chat.

The workflow renders as a **13-node graph**:
`ContextEnricher → 10 risk sub-agents (fan-out) → RiskAggregator (fan-in) → PACO`

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import socket
import threading
from agent_framework.devui import serve

# Check if port 8080 is already in use; pick 8081 if so
def _port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

devui_port = 8080 if not _port_in_use(8080) else 8081

# nest_asyncio patches asyncio.run globally, but uvicorn (used by DevUI)
# calls asyncio.run with a `loop_factory` kwarg that the patched version
# doesn't support. Restore the original asyncio.run inside the DevUI thread
# (which has its own event loop and doesn't need the nest_asyncio patch).
import asyncio.runners
_original_asyncio_run = asyncio.runners.run


def _run_devui():
    asyncio.run = _original_asyncio_run

    # IMPORTANT: rebuild the workflow inside this thread so any asyncio
    # primitives (queues, events) get bound to DevUI's event loop rather
    # than the notebook kernel's loop. Reusing the workflow object that
    # was already run in the notebook causes:
    #   "<Queue ...> is bound to a different event loop"
    devui_workflow = (
        WorkflowBuilder(
            name="ImpossibleTravelInvestigation",
            start_executor=context_enricher,
            output_from=[paco_executor],
        )
        .add_fan_out_edges(context_enricher, ALL_RISK_EXECUTORS)
        .add_fan_in_edges(ALL_RISK_EXECUTORS, risk_aggregator)
        .add_edge(risk_aggregator, paco_executor)
        .build()
    )

    entities = [
        devui_workflow,           # Workflow graph visualization
        *ALL_RISK_AGENTS,         # Individual agents for chat testing
        paco_agent,               # PACO for direct testing
    ]
    serve(entities=entities, port=devui_port, auto_open=False, auth_enabled=False)


devui_thread = threading.Thread(target=_run_devui, daemon=True)
devui_thread.start()

print(f"✅ DevUI running at: http://localhost:{devui_port}")
print("")
print("Available in DevUI:")
print("  📊 ImpossibleTravelInvestigation — workflow graph")
for a in ALL_RISK_AGENTS:
    print(f"  💬 {a.name}")
print("  💬 PACO")

✅ DevUI running at: http://localhost:8080

Available in DevUI:
  📊 ImpossibleTravelInvestigation — workflow graph
  💬 LocationAnomalyAgent
  💬 AuthenticationAnomalyAgent
  💬 TokenAnomalyAgent
  💬 BruteForcePatternAgent
  💬 MfaAbuseAgent
  💬 PostLoginActivityAgent
  💬 OAuthAbuseAgent
  💬 MailboxManipulationAgent
  💬 HighPrivilegeUserAgent
  💬 DisabledAccountSignInAgent
  💬 PACO


INFO:     Started server process [38260]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)


INFO:     127.0.0.1:61532 - "POST /v1/responses HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53103 - "GET /meta HTTP/1.1" 200 OK
INFO:     127.0.0.1:53103 - "GET /v1/entities HTTP/1.1" 200 OK
INFO:     127.0.0.1:53103 - "GET /v1/entities/workflow_in_memory_impossibletravelinvestigation_247dd246afc842d2acd23bc576bda50e/info?type=workflow HTTP/1.1" 200 OK
INFO:     127.0.0.1:53103 - "GET /v1/conversations?entity_id=workflow_in_memory_impossibletravelinvestigation_247dd246afc842d2acd23bc576bda50e&type=workflow_session HTTP/1.1" 200 OK
INFO:     127.0.0.1:53103 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:     127.0.0.1:63022 - "POST /v1/responses HTTP/1.1" 200 OK
  🔍 Context Enricher:  | Dept= | HighPriv=False | Disabled=False
  📊 Risk Aggregator: 8 evidence records, 4 active / 10 total risk factors
